In [18]:
import csv
import ollama
import random

In [11]:
# 例文がある単語をtempに出力
output_csv_path = 'temp'
with open('quiz01-data.txt', mode='r', newline='', encoding='utf-8') as infile:
    reader = csv.reader(infile)        
    headers = next(reader)
    if len(headers) > 2 and headers[2] != '': # 三番目の列が存在するか確認
        with open(output_csv_path, mode='w', newline='', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(headers)
            for row in reader:
                if len(row) > 2:
                    writer.writerow(row)

In [3]:
# 例文がない単語をtempに出力
output_csv_path = 'temp'
with open('quiz01-data.txt', mode='r', newline='', encoding='utf-8') as infile:
    reader = csv.reader(infile)
    headers = next(reader)
    if len(headers) > 2:
        with open(output_csv_path, mode='w', newline='', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(headers)
            for row in reader:
                if len(row) <= 2 or row[2] == '':
                    writer.writerow(row)

In [95]:
options = {
    "num_ctx": 4096,
    "num_predict": 2048,   # 512 → 大幅増
    "temperature": 0.7,
    "top_p": 0.9,
    "repeat_penalty": 1.1,
}
s = """
あなたは韓国語教師です。以下の条件で8単語以内の例文を作成してください。
【使用する単語】: {words}
【使用する文法】: {method}

条件:
- 使用する単語1つにつき、文法「{method}」を使った1つの例文を作成すること
- 使用する単語1つを選んだら使用する単語に関連する語だけから例文を作成すること
- 使用する単語に含まれる他の単語は例文に絶対に使わないこと
- 韓国語例文の後に、意味が完全に一致する日本語訳をつけること
"""

In [104]:
# Qwen3 で例文を作成（quiz01-data.txtから例文がない単語をランダムで個抽出）
method = "한 탓에"
words = ""
with open('quiz01-data.txt', mode='r', newline='', encoding='utf-8') as infile:
    reader = csv.reader(infile)        
    headers = next(reader)
    rows = list(reader)
for row in random.sample(rows,3):
    if len(row) == 2:
        words = words + row[0] + " " + row[1] + ", "
prompt = s.format(words=words, method=method)

print(words)
print(method)

response = ollama.generate(model="qwen3:8b", prompt=prompt, think=False, options=options, keep_alive="30m")
print(response["response"])

서론 序論, 후반전 後半戦, 환기 換気, 
한 탓에
1. 서론을 잘 썼다면 한 탓에 논문이 좋은 점수를 받을 수 있다.  
→ はじめにうまく書ければ、論文の評価が良い点数になるかもしれない。

2. 후반전에 집중하면 한 탓에 승리할 수 있다.  
→ 後半戦に集中すれば、勝つことができるかもしれない。

3. 환기를 하지 않으면 한 탓에 감기 걸릴 수 있다.  
→ 换気をしなければ、風邪を引く可能性がある。


コメントがある単語をlocalStrageにセットする

localStorage.setItem('quiz01-progress-v1', JSON.stringify({"score":{"ok":1,"ng":1,"total":2},"outcomes":{"개선":"ng","과정":"ok"}}));